# Project FORESIGHT — Notebook 03: Forecasting Model vs Baseline

Builds a single **global** model (HistGradientBoostingRegressor) trained across all 50 SKUs,
using only information legitimately available at forecast time, and compares it against the
seasonal-naive baseline from notebook 02 using the same rolling-origin backtest windows.

**Direct multi-horizon strategy**: a separate model is trained for each horizon step
(1..6 weeks ahead). Each model predicts `Units_Sold` at week `t+h` using only features
computed from data up to and including week `t` — this avoids the need to recursively feed
predicted values back in as lags, and keeps every feature causally valid.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor

PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
HORIZON = 6
SEASON_LENGTH = 52
LAGS = [1, 2, 3, 4, 8, 52]
ROLL_WINDOWS = [4, 8]

weekly = pd.read_csv(PROCESSED_DIR / "weekly_demand_panel.csv", parse_dates=["week_start", "Launch_Date"])
weekly = weekly.sort_values(["SKU", "week_index"]).reset_index(drop=True)
weekly.shape

(5250, 17)

## 1. Feature Engineering (Leakage-Safe)

Every lag and rolling-statistic feature is built from `Units_Sold.shift(1)` onward per SKU, so
a feature at week `t` never contains information from week `t` itself or later. Calendar
attributes (month/quarter/season/holiday/promo event) are legitimately known ahead of time
since the calendar is fixed in advance.

In [2]:
def add_features(df):
    df = df.copy()
    g = df.groupby("SKU")["Units_Sold"]
    for lag in LAGS:
        df[f"lag_{lag}"] = g.shift(lag)
    for w in ROLL_WINDOWS:
        # shift(1) first so the rolling window never includes the current week
        df[f"roll_mean_{w}"] = g.shift(1).rolling(w).mean().reset_index(level=0, drop=True)
        df[f"roll_std_{w}"] = g.shift(1).rolling(w).std().reset_index(level=0, drop=True)
    df["age_weeks"] = ((df["week_start"] - df["Launch_Date"]).dt.days / 7).clip(lower=0)
    return df

weekly = add_features(weekly)

CAT_COLS = ["SKU", "Category", "Subcategory", "season"]
for c in CAT_COLS:
    weekly[c] = weekly[c].astype("category")
weekly["month"] = weekly["month"].astype("category")
weekly["quarter"] = weekly["quarter"].astype("category")

FEATURE_COLS = ([f"lag_{l}" for l in LAGS] + [f"roll_mean_{w}" for w in ROLL_WINDOWS] +
                [f"roll_std_{w}" for w in ROLL_WINDOWS] +
                ["is_holiday_week", "has_promo_event", "age_weeks", "Gross_Margin_Per_Unit",
                 "month", "quarter", "season", "Category", "Subcategory", "SKU"])

# Direct multi-horizon targets: target_h at week t = actual demand at week t+h
for h in range(1, HORIZON + 1):
    weekly[f"target_{h}"] = weekly.groupby("SKU")["Units_Sold"].shift(-h)

print(f"Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}")

Feature columns (20): ['lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_52', 'roll_mean_4', 'roll_mean_8', 'roll_std_4', 'roll_std_8', 'is_holiday_week', 'has_promo_event', 'age_weeks', 'Gross_Margin_Per_Unit', 'month', 'quarter', 'season', 'Category', 'Subcategory', 'SKU']


## 2. Rolling-Origin Backtest — Model vs Baseline

Same origins as notebook 02 (non-overlapping 6-week windows) so results are directly comparable.
For each origin and each horizon step `h`, a fresh model is trained only on rows whose target
was already observed by that origin (`week_index + h <= origin`), then used to predict the row
at `week_index == origin`.

In [3]:
def wape(actual, pred):
    actual, pred = np.asarray(actual, dtype=float), np.asarray(pred, dtype=float)
    return np.abs(pred - actual).sum() / actual.sum()

def bias(actual, pred):
    actual, pred = np.asarray(actual, dtype=float), np.asarray(pred, dtype=float)
    return (pred - actual).sum() / actual.sum()

max_week = weekly["week_index"].max()
origin_min = 52
origin_max = max_week - HORIZON
origins = list(range(origin_min, origin_max + 1, HORIZON))
print(f"Backtest origins: {origins}")

Backtest origins: [52, 58, 64, 70, 76, 82, 88, 94]


In [4]:
def fit_predict_horizon(weekly, origin, h):
    """Train a model for horizon h using only data observed by `origin`, predict at `origin`."""
    target_col = f"target_{h}"
    train = weekly[(weekly["week_index"] <= origin - h)].dropna(subset=FEATURE_COLS + [target_col])
    predict_row = weekly[weekly["week_index"] == origin].dropna(subset=FEATURE_COLS)
    if train.empty or predict_row.empty:
        return None

    cat_mask = [c in CAT_COLS + ["month", "quarter"] for c in FEATURE_COLS]
    model = HistGradientBoostingRegressor(
        max_depth=6, learning_rate=0.08, max_iter=200,
        categorical_features=cat_mask, random_state=42,
    )
    model.fit(train[FEATURE_COLS], train[target_col])
    preds = model.predict(predict_row[FEATURE_COLS])
    out = predict_row[["SKU", "week_index"]].copy()
    out["horizon"] = h
    out["target_week_index"] = origin + h
    out["model_pred"] = preds
    return out

model_backtest_rows = []
for origin in origins:
    for h in range(1, HORIZON + 1):
        res = fit_predict_horizon(weekly, origin, h)
        if res is not None:
            model_backtest_rows.append(res)

model_preds = pd.concat(model_backtest_rows, ignore_index=True)
model_preds = model_preds.merge(
    weekly[["SKU", "week_index", "Units_Sold", "baseline_pred", "week_start"]],
    left_on=["SKU", "target_week_index"], right_on=["SKU", "week_index"],
    suffixes=("_origin", ""))
model_preds = model_preds.rename(columns={"week_index": "target_week_index_check"})
print(f"Model backtest predictions: {len(model_preds)} rows across {len(origins)} origins x {HORIZON} horizons")
model_preds.head(3)

Model backtest predictions: 2100 rows across 8 origins x 6 horizons


,SKU,week_index_origin,horizon,target_week_index,model_pred,target_week_index_check,Units_Sold,baseline_pred,week_start
0,SKU001,58,1,59,105.107404,59,116,120.0,2025-02-17
1,SKU002,58,1,59,77.896347,59,82,64.0,2025-02-17
2,SKU003,58,1,59,30.521742,59,45,44.0,2025-02-17


## 3. Baseline vs Model — Backtest Comparison

In [5]:
# Per-origin window metrics for the model (aligning target_week_index back to its origin)
model_preds["origin_week_index"] = model_preds["target_week_index"] - model_preds["horizon"]

comparison_rows = []
for origin in origins:
    mp = model_preds[model_preds["origin_week_index"] == origin]
    bp = weekly[(weekly["week_index"] > origin) & (weekly["week_index"] <= origin + HORIZON)]
    if mp.empty or bp.empty:
        continue
    comparison_rows.append({
        "origin_week_index": origin,
        "test_start": pd.Timestamp(bp["week_start"].min()).date(),
        "baseline_WAPE": wape(bp["Units_Sold"], bp["baseline_pred"]),
        "model_WAPE": wape(mp["Units_Sold"], mp["model_pred"]),
        "baseline_Bias": bias(bp["Units_Sold"], bp["baseline_pred"]),
        "model_Bias": bias(mp["Units_Sold"], mp["model_pred"]),
    })

window_comparison = pd.DataFrame(comparison_rows)
window_comparison

,origin_week_index,test_start,baseline_WAPE,model_WAPE,baseline_Bias,model_Bias
0,58,2025-02-17,0.091368,0.129362,-0.007024,-0.072613
1,64,2025-03-31,0.104157,0.173333,0.004858,0.157549
2,70,2025-05-12,0.093435,0.084386,0.006193,0.033946
3,76,2025-06-23,0.111480,0.172829,0.001573,0.148270
4,82,2025-08-04,0.118818,0.184574,0.004823,0.161812
5,88,2025-09-15,0.115163,0.228264,0.001897,0.205337
6,94,2025-10-27,0.111420,0.114634,-0.001160,-0.006650


In [6]:
overall_baseline_wape = wape(
    weekly.loc[(weekly["week_index"] > origin_min) & (weekly["week_index"] <= origin_max + HORIZON), "Units_Sold"],
    weekly.loc[(weekly["week_index"] > origin_min) & (weekly["week_index"] <= origin_max + HORIZON), "baseline_pred"])
overall_baseline_bias = bias(
    weekly.loc[(weekly["week_index"] > origin_min) & (weekly["week_index"] <= origin_max + HORIZON), "Units_Sold"],
    weekly.loc[(weekly["week_index"] > origin_min) & (weekly["week_index"] <= origin_max + HORIZON), "baseline_pred"])

overall_model_wape = wape(model_preds["Units_Sold"], model_preds["model_pred"])
overall_model_bias = bias(model_preds["Units_Sold"], model_preds["model_pred"])

improvement_pct = 100 * (overall_baseline_wape - overall_model_wape) / overall_baseline_wape
model_beats_baseline = overall_model_wape < overall_baseline_wape

comparison_table = pd.DataFrame([
    {"Method": "Seasonal-Naive Baseline", "WAPE": overall_baseline_wape, "Bias": overall_baseline_bias},
    {"Method": "HistGradientBoosting (global model)", "WAPE": overall_model_wape, "Bias": overall_model_bias},
])
comparison_table["Improvement_vs_Baseline_%"] = [0.0, improvement_pct]

print(f"Baseline WAPE: {overall_baseline_wape:.4f}   Bias: {overall_baseline_bias:+.4f}")
print(f"Model WAPE:    {overall_model_wape:.4f}   Bias: {overall_model_bias:+.4f}")
print(f"Improvement over baseline: {improvement_pct:+.1f}%")
print(f"Model beats baseline: {model_beats_baseline}")
comparison_table

Baseline WAPE: 0.1180   Bias: +0.0136
Model WAPE:    0.1520   Bias: +0.0832
Improvement over baseline: -28.8%
Model beats baseline: False


,Method,WAPE,Bias,Improvement_vs_Baseline_%
0,Seasonal-Naive Baseline,0.118005,0.013618,0.000000
1,HistGradientBoosting (global model),0.151990,0.083196,-28.799275


## 4. Selecting the Forecasting Method

Following the engagement's non-negotiable rule: the model is only adopted if it honestly beats
the baseline on the backtest. Otherwise the baseline is kept and the result reported as-is.

In [7]:
SELECTED_METHOD = "model" if model_beats_baseline else "baseline"
print(f"Selected method for production forecasting: {SELECTED_METHOD.upper()}")
if not model_beats_baseline:
    print("The global HistGradientBoosting model did NOT beat the seasonal-naive baseline "
          "on this backtest. Per the engagement's methodology, the baseline is kept rather than "
          "reporting a fabricated improvement.")

Selected method for production forecasting: BASELINE
The global HistGradientBoosting model did NOT beat the seasonal-naive baseline on this backtest. Per the engagement's methodology, the baseline is kept rather than reporting a fabricated improvement.


## 5. Visuals

In [8]:
fig = go.Figure()
fig.add_trace(go.Bar(x=window_comparison["test_start"].astype(str), y=window_comparison["baseline_WAPE"],
                      name="Baseline WAPE"))
fig.add_trace(go.Bar(x=window_comparison["test_start"].astype(str), y=window_comparison["model_WAPE"],
                      name="Model WAPE"))
fig.update_layout(title="Baseline vs Model WAPE by Backtest Window", barmode="group",
                   xaxis_title="Test window start", yaxis_title="WAPE", template="plotly_white")
fig.show()

In [9]:
# Actual vs baseline vs model for representative SKUs, over the backtest period only
sku_totals = weekly.groupby("SKU")["Units_Sold"].sum().sort_values(ascending=False)
rep_skus = [sku_totals.index[0], sku_totals.index[len(sku_totals)//2], sku_totals.index[-1]]

backtest_period = weekly[(weekly["week_index"] > origin_min) & (weekly["week_index"] <= origin_max + HORIZON)]

fig = go.Figure()
for sku in rep_skus:
    actual_sub = backtest_period[backtest_period["SKU"] == sku]
    model_sub = model_preds[model_preds["SKU"] == sku].sort_values("target_week_index")
    fig.add_trace(go.Scatter(x=actual_sub["week_start"], y=actual_sub["Units_Sold"],
                              mode="lines", name=f"{sku} actual", legendgroup=sku))
    fig.add_trace(go.Scatter(x=actual_sub["week_start"], y=actual_sub["baseline_pred"],
                              mode="lines", name=f"{sku} baseline", line=dict(dash="dot"), legendgroup=sku))
    fig.add_trace(go.Scatter(x=model_sub["week_start"], y=model_sub["model_pred"],
                              mode="lines", name=f"{sku} model", line=dict(dash="dash"), legendgroup=sku))

fig.update_layout(title="Actual vs Baseline vs Model — Backtest Period (representative SKUs)",
                   xaxis_title="Week", yaxis_title="Units Sold", template="plotly_white", height=500)
fig.show()

In [10]:
# Per-SKU WAPE distribution: baseline vs model
per_sku_baseline = backtest_period.groupby("SKU").apply(
    lambda d: wape(d["Units_Sold"], d["baseline_pred"])).rename("Baseline_WAPE")
per_sku_model = model_preds.groupby("SKU").apply(
    lambda d: wape(d["Units_Sold"], d["model_pred"])).rename("Model_WAPE")
per_sku_wape = pd.concat([per_sku_baseline, per_sku_model], axis=1).reset_index()

fig = go.Figure()
fig.add_trace(go.Histogram(x=per_sku_wape["Baseline_WAPE"], name="Baseline", opacity=0.6, nbinsx=15))
fig.add_trace(go.Histogram(x=per_sku_wape["Model_WAPE"], name="Model", opacity=0.6, nbinsx=15))
fig.update_layout(title="Distribution of Per-SKU WAPE: Baseline vs Model", barmode="overlay",
                   xaxis_title="WAPE", yaxis_title="Number of SKUs", template="plotly_white")
fig.show()

In [11]:
fig = px.bar(comparison_table, x="Method", y="Bias", title="Forecast Bias: Baseline vs Model",
             labels={"Bias": "Bias (positive = over-forecast, negative = under-forecast)"},
             template="plotly_white", color="Method")
fig.add_hline(y=0, line_color="black")
fig.show()

In [12]:
# Executive summary visual: WAPE and improvement in one view
fig = go.Figure()
fig.add_trace(go.Bar(x=comparison_table["Method"], y=comparison_table["WAPE"], name="WAPE",
                      marker_color=["lightgray", "steelblue"]))
fig.update_layout(title=f"Forecast Accuracy Summary — {'Model' if model_beats_baseline else 'Baseline'} "
                         f"Selected for Production (WAPE {min(overall_baseline_wape, overall_model_wape):.3f})",
                   yaxis_title="WAPE (lower is better)", template="plotly_white")
fig.show()

## 6. Final Forecast Output (Next 6 Weeks Beyond Available History)

The selected method is used to produce a genuine out-of-sample forecast for the 6 weeks
immediately following the last available week of data, for every SKU.

In [13]:
last_week = max_week

# Week-index -> week_start lookup (built fresh in this notebook, not relying on notebook 02's state)
week_start_by_index = weekly.drop_duplicates("week_index").set_index("week_index")["week_start"]
last_date = week_start_by_index.loc[last_week]

final_baseline_rows = []
final_model_rows = []
for h in range(1, HORIZON + 1):
    target_week_idx = last_week - SEASON_LENGTH + h if (last_week - SEASON_LENGTH + h) in week_start_by_index.index else None
    forecast_date = last_date + pd.Timedelta(weeks=h)

    # Baseline: seasonal value from 52 weeks before the forecasted week, per SKU
    if target_week_idx is not None:
        base_vals = weekly[weekly["week_index"] == target_week_idx][["SKU", "Units_Sold"]].rename(
            columns={"Units_Sold": "baseline_prediction"})
    else:
        base_vals = weekly[weekly["week_index"] == last_week][["SKU", "Units_Sold"]].rename(
            columns={"Units_Sold": "baseline_prediction"})
    base_vals["forecast_week"] = forecast_date
    final_baseline_rows.append(base_vals)

    # Model: train on ALL available history for this horizon, predict from the last known week
    target_col = f"target_{h}"
    train = weekly.dropna(subset=FEATURE_COLS + [target_col])
    predict_row = weekly[weekly["week_index"] == last_week].dropna(subset=FEATURE_COLS)
    if not train.empty and not predict_row.empty:
        cat_mask = [c in CAT_COLS + ["month", "quarter"] for c in FEATURE_COLS]
        model = HistGradientBoostingRegressor(max_depth=6, learning_rate=0.08, max_iter=200,
                                               categorical_features=cat_mask, random_state=42)
        model.fit(train[FEATURE_COLS], train[target_col])
        preds = model.predict(predict_row[FEATURE_COLS])
        m = predict_row[["SKU"]].copy()
        m["model_prediction"] = preds
        m["forecast_week"] = forecast_date
        final_model_rows.append(m)

final_baseline = pd.concat(final_baseline_rows, ignore_index=True)
final_model = pd.concat(final_model_rows, ignore_index=True) if final_model_rows else pd.DataFrame(
    columns=["SKU", "model_prediction", "forecast_week"])

forecast_output = final_baseline.merge(final_model, on=["SKU", "forecast_week"], how="left")
forecast_output["predicted_demand"] = (
    forecast_output["model_prediction"] if model_beats_baseline else forecast_output["baseline_prediction"])
forecast_output["selected_model"] = "HistGradientBoosting" if model_beats_baseline else "Seasonal-Naive Baseline"
forecast_output = forecast_output.rename(columns={"baseline_prediction": "baseline_prediction"})
forecast_output = forecast_output[["SKU", "forecast_week", "predicted_demand", "baseline_prediction",
                                    "model_prediction", "selected_model"]]
forecast_output = forecast_output.sort_values(["SKU", "forecast_week"]).reset_index(drop=True)
forecast_output.head(10)

,SKU,forecast_week,predicted_demand,baseline_prediction,model_prediction,selected_model
0,SKU001,2026-01-05,100,100,82.676001,Seasonal-Naive Baseline
1,SKU001,2026-01-12,89,89,57.441444,Seasonal-Naive Baseline
2,SKU001,2026-01-19,103,103,65.394641,Seasonal-Naive Baseline
3,SKU001,2026-01-26,100,100,55.741381,Seasonal-Naive Baseline
4,SKU001,2026-02-02,106,106,94.877820,Seasonal-Naive Baseline
5,SKU001,2026-02-09,124,124,94.507256,Seasonal-Naive Baseline
6,SKU002,2026-01-05,63,63,51.391723,Seasonal-Naive Baseline
7,SKU002,2026-01-12,67,67,53.681716,Seasonal-Naive Baseline
8,SKU002,2026-01-19,66,66,46.455950,Seasonal-Naive Baseline
9,SKU002,2026-01-26,85,85,47.927406,Seasonal-Naive Baseline


## 7. Save Deliverables

In [14]:
forecast_output.to_csv(PROCESSED_DIR / "forecast_output.csv", index=False)

metrics_table = comparison_table.copy()
metrics_table.to_csv(REPORTS_DIR / "forecast_metrics.csv", index=False)

with open(REPORTS_DIR / "forecast_summary.txt", "w") as f:
    f.write("FORESIGHT — FORECAST MODEL SUMMARY\n")
    f.write("=" * 40 + "\n\n")
    f.write(f"Forecast horizon: {HORIZON} weeks\n")
    f.write(f"Backtest windows: {len(origins)} (rolling-origin, non-overlapping)\n\n")
    f.write(f"Baseline (seasonal-naive) WAPE: {overall_baseline_wape:.4f}, Bias: {overall_baseline_bias:+.4f}\n")
    f.write(f"Model (HistGradientBoosting)  WAPE: {overall_model_wape:.4f}, Bias: {overall_model_bias:+.4f}\n")
    f.write(f"Improvement over baseline: {improvement_pct:+.1f}%\n")
    f.write(f"Model beats baseline: {model_beats_baseline}\n")
    f.write(f"Selected method for production: {forecast_output['selected_model'].iloc[0]}\n\n")
    if not model_beats_baseline:
        f.write("Finding: the global gradient-boosting model did not beat the seasonal-naive "
                "baseline on this backtest. The baseline is used for the forecast_output.csv "
                "rather than reporting a fabricated improvement.\n")

print("Saved:")
print(" - data/processed/forecast_output.csv")
print(" - reports/forecast_metrics.csv")
print(" - reports/forecast_summary.txt")

Saved:
 - data/processed/forecast_output.csv
 - reports/forecast_metrics.csv
 - reports/forecast_summary.txt
